# run_calibrate: camera intrinsics from ChArUco views (M1)

Needs the real camera and the printed ChArUco board (`markers.ipynb`). Not executed in CI.
Writes `calibration/camera_intrinsics.yaml` and prints the reprojection-error report.

**TODO**
- [ ] implement `collect_views` / `coverage_report` in `calibration.ipynb`
- [ ] set `camera.device` in `configs/camera.yaml`
- [ ] accept the calibration only if the RMS error is low (e.g. < 0.5 px); record it in the report

In [ ]:
%run ./calibration.ipynb
%cd ..
import cv2
import numpy as np

cam_cfg = load_config("configs/camera.yaml")
board = make_charuco_board(cam_cfg.charuco)

## 1. Collect views
Hold the board still at many positions (all image corners, tilted). Aim for 20-30 views.

In [ ]:
cap = cv2.VideoCapture(cam_cfg.camera.device)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, cam_cfg.camera.width)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cam_cfg.camera.height)
try:
    frames, views = collect_views(cap, board, n_views=25, min_corners=cam_cfg.charuco.min_corners)  # TODO M1
finally:
    cap.release()
image_size = (frames[0].shape[1], frames[0].shape[0])
print(len(views), "views", image_size)
# print(coverage_report(views, image_size))   # TODO M1

## 2. Calibrate and report

In [ ]:
result = calibrate_charuco(views, board, image_size)
print(f"RMS reprojection error: {result.rms:.3f} px over {len(result.per_view_errors)} views")
print("K =\n", result.camera_matrix, "\ndist =", result.dist_coeffs)
worst = np.argsort(result.per_view_errors)[::-1][:5]
print("worst views:", [(int(i), round(result.per_view_errors[i], 3)) for i in worst])
# TODO M1: drop outlier views (e.g. error > 2x median) and recalibrate.

## 3. Save and check undistortion visually

In [ ]:
path = save_intrinsics(result)
print("wrote", path)
und = undistort(frames[0], result.camera_matrix, result.dist_coeffs)
cv2.imwrite("calibration/undistort_check.png", np.hstack([frames[0], und]))